In [1]:
import zipfile
import pandas as pd
from dataclasses import dataclass
from pathlib import Path
from cnnClassifier import logger
from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml, create_directories, get_size
from sklearn.model_selection import train_test_split

I0000 00:00:1793790935.633751   88123 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
%pwd

'/home/nima/Projects/ML-Project-Casting_Classification/Project/research'

In [3]:
import os
os.chdir("../")

In [4]:
%pwd

'/home/nima/Projects/ML-Project-Casting_Classification/Project'

In [ ]:
@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path
    dataset_dir: Path
    manifest_path: Path

@dataclass(frozen=True)
class KaggleToken:
    username: str
    key: str

@dataclass
class ImageData:
    CLASS_TO_IDX: dict
    IMG_EXTNS: list

In [8]:
class ConfigurationManager:
    def __init__(
        self,
        secrets_filepath = TOKEN_KEY_PATH,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):

        self.secrets = read_yaml(secrets_filepath)
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])


    
    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion

        create_directories([config.root_dir])

        data_ingestion_config = DataIngestionConfig(
            root_dir=Path(config.root_dir),
            source_URL=config.source_URL,
            local_data_file=Path(config.local_data_file),
            unzip_dir=Path(config.unzip_dir),
            dataset_dir=Path(config.dataset_dir),
            manifest_path=Path(config.manifest_path)
        )

        return data_ingestion_config

    
    def get_data_ingestion_secrets(self) -> KaggleToken:
        secrets = self.secrets.kaggle_secrets

        data_ingestion_secrets = KaggleToken(
            username=secrets.username,
            key=secrets.key
        )

        return data_ingestion_secrets
    
    
    def get_data_ingestion_cls_idx(self) -> ImageData:
        params = self.params

        data_ingestion_cls_idx = ImageData(
            CLASS_TO_IDX = params.CLASS_TO_IDX,
            IMG_EXTNS = params.IMG_EXTNS
        )

        return data_ingestion_cls_idx

In [9]:
class DataIngestion:
    def __init__(self, config: DataIngestionConfig, token: KaggleToken, img_class_extns: ImageData):
        self.config = config
        self.secrets = token
        self.img_class_extns = img_class_extns
    
    def download_file(self):
        if not os.path.exists(self.config.local_data_file):
            os.environ['KAGGLE_USERNAME'] = self.secrets.username
            os.environ['KAGGLE_KEY'] = self.secrets.key
            import kaggle 
            url = self.config.source_URL            
            kaggle.api.authenticate()
            kaggle.api.dataset_download_files(url,
                                             path=self.config.unzip_dir,
                                             unzip=False)
            
            logger.info(f" Casting dataset download is executed!")
        else:
            logger.info(f"File already exist and has a size of: {get_size(Path(self.config.local_data_file))}")


    
    def extract_zip_file(self):
        """
        zip_file_path: str
        Extracts the zip file into the data directory
        Function returns None
        """
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
            zip_ref.extractall(unzip_path)
            logger.info(f"Downloaded zipfile is extracted")


    def collect_filepaths(self) -> pd.DataFrame:
        """Scan class subfolders and return a DataFrame with filepath, class name and label."""
        records = []
        for class_name, label in self.img_class_extns.CLASS_TO_IDX.items():
            class_dir = Path(os.path.join(self.config.dataset_dir, class_name))
            if not class_dir.is_dir():
                raise FileNotFoundError(f"Class folder not found: {class_dir}")
            for fp in sorted(class_dir.iterdir()):  # sorted -> reproducible order
                if fp.suffix.lower() in self.img_class_extns.IMG_EXTNS:
                    records.append({"filepath": str(fp), "class_name": class_name, "label": label})
        logger.info("data_set has been read into a pandas dataframe")
        return pd.DataFrame(records)

    @staticmethod
    def stratified_train_val_test_split(                                        
                                        df: pd.DataFrame,
                                        train_ratio: float = 0.8,
                                        val_ratio: float = 0.1,
                                        test_ratio: float = 0.1,
                                        label_col: str = "label",
                                        seed: int = 42,
                                        ):

        """Stratified split into train/val/test using two successive splits."""
        if abs(train_ratio + val_ratio + test_ratio - 1.0) > 1e-9:
            raise ValueError("Ratios must sum to 1.0")

        train_df, temp_df = train_test_split(
            df,
            test_size=(val_ratio + test_ratio),
            stratify=df[label_col],
            random_state=seed,
        )
        logger.info("train_set is created")

        # share of the temp set that becomes the test set
        test_share = test_ratio / (val_ratio + test_ratio)
        val_df, test_df = train_test_split(
            temp_df,
            test_size=test_share,
            stratify=temp_df[label_col],
            random_state=seed,
        )
        logger.info("test_set val_set are created")
        return (
                train_df.reset_index(drop=True),
                val_df.reset_index(drop=True),
                test_df.reset_index(drop=True),
               )

    def save_split_manifests(self, train_df, val_df, test_df) -> None:
        out_dir = self.config.manifest_path
        out_dir.mkdir(parents=True, exist_ok=True)
        for name, part in [("train", train_df), ("val", val_df), ("test", test_df)]:
            part.to_csv(out_dir / f"{name}.csv", index=False)
        logger.info(f"Manifests saved to {out_dir.resolve()}")

In [10]:
try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion_secrets = config.get_data_ingestion_secrets()
    get_data_ingestion_cls_idx = config.get_data_ingestion_cls_idx()
    
    data_ingestion = DataIngestion(config=data_ingestion_config,
                                   token=data_ingestion_secrets,
                                   img_class_extns=get_data_ingestion_cls_idx
                                   )
    
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()
    dataframe = data_ingestion.collect_filepaths()
    train_df, val_df, test_df = data_ingestion.stratified_train_val_test_split(dataframe)
    data_ingestion.save_split_manifests(train_df, val_df, test_df)

except Exception as e:
    raise e

[2026-11-04 12:15:38,185: INFO: common: yaml file: token_keys/secret.yaml loaded successfully]
[2026-11-04 12:15:38,189: INFO: common: yaml file: config/config.yaml loaded successfully]
[2026-11-04 12:15:38,192: INFO: common: yaml file: params.yaml loaded successfully]
[2026-11-04 12:15:38,193: INFO: common: created directory at: artifacts]
[2026-11-04 12:15:38,194: INFO: common: created directory at: artifacts/data_ingestion]
Dataset URL: https://www.kaggle.com/datasets/ravirajsinh45/real-life-industrial-dataset-of-casting-product
[2026-11-04 12:15:43,148: INFO: 2652132684:  Casting dataset download is executed!]
[2026-11-04 12:15:44,474: INFO: 2652132684: Downloaded zipfile is extracted]
[2026-11-04 12:15:44,486: INFO: 2652132684: data_set has been read into a pandas dataframe]
[2026-11-04 12:15:44,531: INFO: 2652132684: train_set is created]
[2026-11-04 12:15:44,534: INFO: 2652132684: test_set val_set are created]
[2026-11-04 12:15:44,542: INFO: 2652132684: Manifests saved to /home/